# FRED + EIA Macroeconomic & Fuel Price Data

## Purpose

Build a daily **macroeconomic context table** for each M5 state. Consumer spending power shifts with unemployment, inflation, gas prices, and sentiment — factors external to Walmart's pricing decisions that affect demand independently of anything in the M5 data.

### Data Sources
- **FRED API**: State unemployment (CA, TX, WI), national CPI, consumer sentiment — monthly
- **EIA API**: Regional gasoline prices by PADD region — weekly

In [ ]:
# ── Uncomment below when running on Google Colab ────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/Colab Notebooks/m5-forecasting-accuracy/notebooks')

BASE_DIR = '..'
RAW_DIR = f'{BASE_DIR}/data/raw'
CLEAN_DIR = f'{BASE_DIR}/data/clean'
FIG_DIR = f'{BASE_DIR}/figures'

import os
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Project dir: {BASE_DIR}')

## Step 0: Setup

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os
import warnings
warnings.filterwarnings('ignore')

# RAW_DIR defined in mount cell above
# CLEAN_DIR defined in mount cell above

DATE_START = '2011-01-29'
DATE_END = '2016-06-19'

try:
    from google.colab import userdata
    FRED_API_KEY = userdata.get('FRED_API_KEY')
    EIA_API_KEY = userdata.get('EIA_API_KEY')
except:
    FRED_API_KEY = os.environ.get('FRED_API_KEY', '')
    EIA_API_KEY = os.environ.get('EIA_API_KEY', '')
assert FRED_API_KEY, 'Set FRED_API_KEY in Colab Secrets or as env var'
assert EIA_API_KEY, 'Set EIA_API_KEY in Colab Secrets or as env var'
print(f'FRED API key: {FRED_API_KEY[:8]}...')
print(f'EIA API key: {EIA_API_KEY[:8]}...')

## Step 1: Fetch FRED series

We pull 5 series from FRED:
- **State unemployment** (CAUR, TXUR, WIUR) — state-level because CA/TX/WI have different economic cycles
- **National CPI** (CPIAUCSL) — inflation affects all states similarly
- **Consumer Sentiment** (UMCSENT) — how consumers *feel* about the economy (leading indicator)

All are monthly. We start from 2010-01-01 (before our M5 range) so we have lookback data for derived features.

In [11]:
FRED_BASE = 'https://api.stlouisfed.org/fred/series/observations'

FRED_SERIES = {
    'CAUR':     {'name': 'CA Unemployment Rate',      'state': 'CA'},
    'TXUR':     {'name': 'TX Unemployment Rate',      'state': 'TX'},
    'WIUR':     {'name': 'WI Unemployment Rate',      'state': 'WI'},
    'CPIAUCSL': {'name': 'CPI All Urban Consumers',   'state': 'ALL'},
    'UMCSENT':  {'name': 'Consumer Sentiment Index',  'state': 'ALL'},
}

def fetch_fred_series(series_id, api_key, start_date, end_date):
    params = {
        'series_id': series_id, 'api_key': api_key,
        'file_type': 'json',
        'observation_start': start_date, 'observation_end': end_date,
    }
    for attempt in range(3):
        try:
            resp = requests.get(FRED_BASE, params=params, timeout=30)
            resp.raise_for_status()
            observations = resp.json().get('observations', [])
            df = pd.DataFrame(observations)
            df['date'] = pd.to_datetime(df['date'])
            df['value'] = pd.to_numeric(df['value'], errors='coerce')
            return df[['date', 'value']].dropna()
        except Exception as e:
            print(f'    Retry {attempt+1}: {e}')
            time.sleep(2)
    return pd.DataFrame(columns=['date', 'value'])

fred_data = {}
for series_id, info in FRED_SERIES.items():
    print(f'Fetching {series_id} ({info["name"]})...', end=' ')
    df = fetch_fred_series(series_id, FRED_API_KEY, '2010-01-01', DATE_END)
    fred_data[series_id] = df
    print(f'{len(df)} observations, range [{df["value"].min():.1f}, {df["value"].max():.1f}]')
    time.sleep(0.3)

Fetching CAUR (CA Unemployment Rate)... 78 observations, range [5.5, 12.4]
Fetching TXUR (TX Unemployment Rate)... 78 observations, range [4.4, 8.4]
Fetching WIUR (WI Unemployment Rate)... 78 observations, range [3.9, 9.2]
Fetching CPIAUCSL (CPI All Urban Consumers)... 78 observations, range [217.2, 240.2]
Fetching UMCSENT (Consumer Sentiment Index)... 78 observations, range [55.8, 98.1]


## Step 2: Fetch EIA gasoline prices

Weekly retail gasoline prices by PADD region:
- **PADD 5** (West Coast) → CA — highest gas taxes + special blend → ~\$0.40/gal premium
- **PADD 3** (Gulf Coast) → TX — near refineries → cheapest gas
- **PADD 2** (Midwest) → WI — middle pricing

**Why regional?** A CA shopper paying \$4.20/gal has very different trip behavior than TX at \$2.80/gal.

In [4]:
EIA_BASE = 'https://api.eia.gov/v2/petroleum/pri/gnd/data/'

PADD_SERIES = {
    'CA': {'id': 'EMM_EPMR_PTE_R50_DPG', 'name': 'West Coast (PADD 5)'},
    'TX': {'id': 'EMM_EPMR_PTE_R30_DPG', 'name': 'Gulf Coast (PADD 3)'},
    'WI': {'id': 'EMM_EPMR_PTE_R20_DPG', 'name': 'Midwest (PADD 2)'},
}

def fetch_eia_gas(api_key, series_id, start_date, end_date):
    params = {
        'api_key': api_key, 'frequency': 'weekly',
        'data[0]': 'value', 'facets[series][]': series_id,
        'start': start_date[:7], 'end': end_date[:7],
        'sort[0][column]': 'period', 'sort[0][direction]': 'asc',
        'length': 5000,
    }
    for attempt in range(3):
        try:
            resp = requests.get(EIA_BASE, params=params, timeout=30)
            resp.raise_for_status()
            rows = resp.json().get('response', {}).get('data', [])
            if rows:
                df = pd.DataFrame(rows)
                df['date'] = pd.to_datetime(df['period'])
                df['gas_price'] = pd.to_numeric(df['value'], errors='coerce')
                return df[['date', 'gas_price']].dropna().sort_values('date')
            return pd.DataFrame(columns=['date', 'gas_price'])
        except Exception as e:
            print(f'    Retry {attempt+1}: {e}')
            time.sleep(2)
    return pd.DataFrame(columns=['date', 'gas_price'])

gas_data = {}
for state, info in PADD_SERIES.items():
    print(f'Fetching {state} gas prices ({info["name"]})...', end=' ')
    df = fetch_eia_gas(EIA_API_KEY, info['id'], '2010-01-01', DATE_END)
    gas_data[state] = df
    if len(df) > 0:
        print(f'{len(df)} weeks, ${df["gas_price"].min():.2f} - ${df["gas_price"].max():.2f}')
    else:
        print('NO DATA')
    time.sleep(0.5)

Fetching CA gas prices (West Coast (PADD 5))... 335 weeks, $2.14 - $4.41
Fetching TX gas prices (Gulf Coast (PADD 3))... 335 weeks, $1.53 - $3.82
Fetching WI gas prices (Midwest (PADD 2))... 335 weeks, $1.52 - $4.01


## Step 3: Combine and forward-fill to daily

FRED is monthly, EIA is weekly — but our sales data is daily. We **forward-fill**: each day inherits the most recent observation.

**Why forward-fill, not interpolation?** When BLS releases January unemployment on Feb 1, the rate *was* January's number all of January. Forward-fill models this correctly.

In [5]:
date_range = pd.date_range(DATE_START, DATE_END, freq='D')
states = ['CA', 'TX', 'WI']

# State x date scaffold
scaffold = pd.MultiIndex.from_product([states, date_range], names=['state_id', 'date'])
macro = pd.DataFrame(index=scaffold).reset_index()

# Merge state unemployment
unemp_map = {'CAUR': 'CA', 'TXUR': 'TX', 'WIUR': 'WI'}
unemp_dfs = []
for series_id, state in unemp_map.items():
    df = fred_data[series_id].copy()
    df['state_id'] = state
    df = df.rename(columns={'value': 'unemployment_state'})
    unemp_dfs.append(df)
unemp = pd.concat(unemp_dfs).sort_values(['state_id', 'date'])
macro = macro.merge(unemp[['state_id', 'date', 'unemployment_state']], on=['state_id', 'date'], how='left')
macro['unemployment_state'] = macro.groupby('state_id')['unemployment_state'].ffill().bfill()

# National indicators (broadcast to all states)
for series_id, col_name in [('CPIAUCSL', 'cpi'), ('UMCSENT', 'consumer_sentiment')]:
    df = fred_data[series_id].rename(columns={'value': col_name})
    df = df.set_index('date').reindex(date_range).ffill().bfill().reset_index()
    df = df.rename(columns={'index': 'date'})
    macro = macro.merge(df[['date', col_name]], on='date', how='left')

# Gas prices
gas_dfs = []
for state, df in gas_data.items():
    df = df.copy()
    df['state_id'] = state
    df = df.rename(columns={'gas_price': 'gas_price_regional'})
    gas_dfs.append(df)
if gas_dfs:
    gas = pd.concat(gas_dfs)
    macro = macro.merge(gas[['state_id', 'date', 'gas_price_regional']], on=['state_id', 'date'], how='left')
    macro['gas_price_regional'] = macro.groupby('state_id')['gas_price_regional'].ffill().bfill()
else:
    macro['gas_price_regional'] = np.nan

print(f'Macro table shape: {macro.shape}')
print(f'Missing values:\n{macro.isnull().sum()}')

Macro table shape: (5907, 6)
Missing values:
state_id              0
date                  0
unemployment_state    0
cpi                   0
consumer_sentiment    0
gas_price_regional    0
dtype: int64


## Step 4: Derived features

Raw numbers need transformation to capture *change*, not just level:
- `cpi_yoy_change`: year-over-year inflation rate (%)
- `unemployment_delta_3m`: 3-month change in unemployment
- `gas_price_change_4wk`: 4-week % change in gas price

**Why percentage change?** A \$0.50 increase from \$2.00 is 25% (significant); from \$4.00 is 12.5% (less impactful). Percentage change normalizes across different price levels.

In [6]:
macro['cpi_yoy_change'] = macro.groupby('state_id')['cpi'].transform(
    lambda x: x.pct_change(periods=365) * 100
)
macro['cpi_yoy_change'] = macro['cpi_yoy_change'].fillna(0)

macro['unemployment_delta_3m'] = macro.groupby('state_id')['unemployment_state'].transform(
    lambda x: x.diff(periods=90)
)
macro['unemployment_delta_3m'] = macro['unemployment_delta_3m'].fillna(0)

macro['gas_price_change_4wk'] = macro.groupby('state_id')['gas_price_regional'].transform(
    lambda x: x.pct_change(periods=28) * 100
)
macro['gas_price_change_4wk'] = macro['gas_price_change_4wk'].fillna(0)

print(f'Final macro shape: {macro.shape}')
print(f'Columns: {list(macro.columns)}')
macro.describe().round(2)

Final macro shape: (5907, 9)
Columns: ['state_id', 'date', 'unemployment_state', 'cpi', 'consumer_sentiment', 'gas_price_regional', 'cpi_yoy_change', 'unemployment_delta_3m', 'gas_price_change_4wk']


,date,unemployment_state,cpi,consumer_sentiment,gas_price_regional,cpi_yoy_change,unemployment_delta_3m,gas_price_change_4wk
count,5907,5907.00,5907.00,5907.00,5907.00,5907.00,5907.00,5907.00
mean,2013-10-09 00:00:00,6.86,232.95,81.14,3.21,1.03,-0.21,-0.18
min,2011-01-29 00:00:00,3.90,221.90,55.80,1.52,-0.23,-0.80,-23.67
25%,2012-06-04 00:00:00,5.30,228.81,75.00,2.78,0.01,-0.30,-4.47
50%,2013-10-09 00:00:00,6.70,233.67,81.90,3.39,1.14,-0.20,-0.64
75%,2015-02-13 00:00:00,7.70,237.34,90.00,3.67,1.69,-0.10,4.00
max,2016-06-19 00:00:00,12.00,240.22,98.10,4.41,2.90,0.30,34.45
std,NaN,2.03,4.69,9.71,0.64,0.84,0.19,6.94


### Interpretation

The derived features exhibit meaningful variation across the 2011–2016 window. **CPI year-over-year change** ranges from -0.23% to 2.90%, capturing the post-recession recovery through moderate inflation. **Unemployment delta (3-month)** is consistently negative (mean = -0.21), reflecting the steady labor market recovery across all three states. **Gas price change** is the most volatile feature (std = 6.94%), with swings from -23.7% to +34.5% — these sharp movements are likely to affect trip consolidation behavior at Walmart stores.

## Step 5: Sanity checks

In [7]:
print('Unemployment by state (should decline 2011->2016):')
for state in states:
    s = macro[macro['state_id'] == state]
    first = s[s['date'] <= '2011-06-30']['unemployment_state'].mean()
    last = s[s['date'] >= '2016-01-01']['unemployment_state'].mean()
    print(f'  {state}: {first:.1f}% (2011H1) -> {last:.1f}% (2016H1)')

print(f'\nGas prices by state (should show CA > WI > TX):')
for state in states:
    avg = macro[macro['state_id'] == state]['gas_price_regional'].mean()
    print(f'  {state}: ${avg:.2f} avg')

print(f'\nCPI range: {macro["cpi"].min():.1f} to {macro["cpi"].max():.1f}')

Unemployment by state (should decline 2011->2016):
  CA: 11.9% (2011H1) -> 5.6% (2016H1)
  TX: 8.2% (2011H1) -> 4.5% (2016H1)
  WI: 7.6% (2011H1) -> 4.1% (2016H1)

Gas prices by state (should show CA > WI > TX):
  CA: $3.53 avg
  TX: $2.97 avg
  WI: $3.13 avg

CPI range: 221.9 to 240.2


### Interpretation

All sanity checks pass. **Unemployment declined from 2011 to 2016** across all three states (CA: 11.9% → 5.6%, TX: 8.2% → 4.5%, WI: 7.6% → 4.1%), consistent with the post-Great Recession recovery. **Gas price ordering** (CA \$3.53 > WI \$3.13 > TX \$2.97) matches expected regional pricing due to refinery proximity and state tax differences.

## Step 6: Save output

**Merge key**: `(state_id, date)` — macro conditions are state-level.

In [8]:
macro.to_csv(f'{CLEAN_DIR}/macro_indicators.csv', index=False)
print(f'Saved: macro_indicators.csv ({len(macro)} rows)')
print(f'Columns: {list(macro.columns)}')
print(f'Merge key for downstream: (state_id, date)')

Saved: macro_indicators.csv (5907 rows)
Columns: ['state_id', 'date', 'unemployment_state', 'cpi', 'consumer_sentiment', 'gas_price_regional', 'cpi_yoy_change', 'unemployment_delta_3m', 'gas_price_change_4wk']
Merge key for downstream: (state_id, date)


---
## Conclusion

This notebook produces `macro_indicators.csv` — daily macro context per state, combining FRED state unemployment (per-state, monthly → forward-filled to daily), national CPI (monthly → forward-filled), and EIA gasoline prices (weekly → forward-filled). Derived features include year-over-year CPI change, 3-month unemployment delta, and 4-week gas price percentage change to capture trend direction at multiple time scales.

**Caveats**: CPI is national even though grouped by state (a no-op groupby); gas prices are national PADD-level, not city-level. State-level granularity is preserved only for unemployment.

**Merge key for downstream**: `(state_id, date)`.